# CareDesk — NileCare Hospital Knowledge Assistant

**Mid-term:** Building a Basic AI Knowledge Assistant with Memory  
**Hospital:** NileCare General Hospital (staff operations only — not medical advice)  
**Model:** `Qwen/Qwen2-1.5B-Instruct`  
**Orchestrator:** LangChain LCEL (one chain of steps)

---

## Dictionary (read this first)

| # | Section | This code does the following |
|---|---------|------------------------------|
| 0 | GPU check | Stops the notebook if there is no GPU |
| 1 | Installs | Installs Transformers, LangChain, FAISS |
| 2 | Imports | Loads every library in one place |
| 3 | Model | Loads Qwen in 4-bit |
| 4 | Generate | Turns a prompt into new tokens only |
| 5 | PDF | Creates and reads the staff policy |
| 6 | Google Sheet | Builds the staff roster table |
| 7 | Chunk + FAISS | Splits text and stores vectors |
| 8 | Retriever chain | LangChain chain: question → closest chunks |
| 9 | Tool chains | Four LangChain tools that read the sheet |
| 10 | Memory | Saves the full chat (buffer memory) |
| 11 | Prompt + answer chains | Quote chain + factual answer builder |
| 12 | LangChain orchestrator | One chain: parse → tools → retrieve → decide → speak → remember |
| 13 | Tests | Allow, deny, escalate, follow-up, not found |
| 14 | Evaluation | Scores decision, tools, memory, relevance |
| 15 | Live chat | Type questions; `quit` to stop |

---

## What CareDesk is

A nurse or HR officer asks about leave, shift coverage, isolation access, or who to email.  
CareDesk answers from a **policy PDF** and a **staff sheet**, remembers the chat, and prints a **decision trace**.

Python decides ALLOW / DENY / ESCALATE / NOT FOUND / INFO.  
The final spoken answer is built from that decision and the tool facts, so the small model cannot contradict them.

---

## Architecture

LangChain is the manager. Each box is one `RunnableLambda` in one LCEL chain.

```
                    LANGCHAIN CHAIN
+--------------------------------------------------+
|  question                                        |
|    -> parse       name, days, which tools        |
|    -> tools       4 LangChain tools (sheet)      |
|    -> retrieve    FAISS + prompt chain (PDF)     |
|    -> decide      Python rules                   |
|    -> speak       build_answer from facts        |
|    -> remember    buffer memory + print trace    |
+--------------------------------------------------+
```

**Colab:** Runtime → Change runtime type → **T4 GPU** → Save, then run from the top.

## 0. GPU check

**This code does the following:** Confirms a GPU is on. The 4-bit model hangs on CPU.

In [1]:
import torch

print("GPU on?", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU → Save → Restart, then run again.")
print("GPU name:", torch.cuda.get_device_name(0))

GPU on? True
GPU name: Tesla T4


## 1. Installs

**This code does the following:** Installs the libraries. There is no LangGraph. LangChain is the orchestrator.

In [2]:
!pip install -q bitsandbytes accelerate transformers faiss-cpu sentence-transformers langchain langchain-core langchain-community langchain-huggingface pypdf fpdf pandas openpyxl

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 2. Imports

**This code does the following:** Loads Transformers, FAISS, and the LangChain pieces used as chains: PromptTemplate, StructuredTool, RunnableLambda.

In [3]:
import re
import json
from datetime import datetime, date

import numpy as np
import pandas as pd
import torch
import faiss
from fpdf import FPDF
from pypdf import PdfReader

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import StructuredTool
from langchain_core.runnables import RunnableLambda

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

## 3. Model loading

**This code does the following:** Loads Qwen2-1.5B-Instruct in 4-bit, the same way as Session 3.

In [4]:
model_name = "Qwen/Qwen2-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

print("Model loaded on", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on cuda:0


## 4. Generate function

**This code does the following:** Sends a prompt and returns only the new tokens. Sampling is off so the model does not repeat the prompt.

In [5]:
def generate(prompt, max_tokens=80):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def llm_text_is_bad(text):
    if not text or len(text) < 8:
        return True
    low = text.lower()
    leaked = ["write 3 to 5", "if decision is not found", "short answer:", "user question:"]
    return any(p in low for p in leaked) or low.count("not found") > 1

## 5. PDF (unstructured data)

**This code does the following:** Creates the NileCare staff policy PDF and reads it back. This is the unstructured knowledge source.

In [6]:
POLICY_TEXT = """
NileCare General Hospital
Staff Operations Policy
Version 1.0
This document is for staff operations only. It is not medical advice.

Section A. Annual Leave
Full-time staff receive 21 annual leave days per year.
A leave request is denied if the requested days are greater than the staff member remaining leave balance.
Staff in the first 90 days of employment are on probation.
Probation staff may request at most 2 leave days and the request must be sent to their department lead.

Section B. Shift Coverage
Each ward has a minimum number of on-shift staff.
ICU minimum is 3. ER minimum is 2. Pediatrics minimum is 2.
If a staff member is on shift today and approving their leave would drop the ward below the minimum, the request is not approved automatically. It must be escalated to the department lead.

Section C. Isolation Access
Only staff with isolation clearance may enter isolation rooms.
Staff without isolation clearance must not be assigned to isolation rooms.
This rule is about room access. It is not a medical treatment protocol.

Section D. Department Contacts
ICU department lead is Head Nurse Mona Farid. Email: icu.lead@nilecare.example
ER department lead is Dr. Hany Aziz. Email: er.lead@nilecare.example
Pediatrics department lead is Head Nurse Nour Adel. Email: peds.lead@nilecare.example
HR contact is Laila Hassan. Email: hr@nilecare.example
""".strip()


def write_policy_pdf(path="nilecare_staff_policy.pdf"):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    for line in POLICY_TEXT.split("\n"):
        pdf.multi_cell(0, 8, line)
    pdf.output(path)
    return path


def load_pdf_text(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        if page.extract_text():
            text += page.extract_text() + "\n"
    return text.strip()


pdf_path = write_policy_pdf()
pdf_text = load_pdf_text(pdf_path)
print("PDF saved:", pdf_path)
print(pdf_text[:350])

PDF saved: nilecare_staff_policy.pdf
NileCare General Hospital
Staff Operations Policy
Version 1.0
This document is for staff operations only. It is not medical advice.
Section A. Annual Leave
Full-time staff receive 21 annual leave days per year.
A leave request is denied if the requested days are greater than the staff member remaining leave
balance.
Staff in the first 90 days of em


## 6. Google Sheet (structured data)

**This code does the following:** Builds the staff roster and saves it as a CSV in the same format as a published Google Sheet.

To use a live Sheet later: upload the CSV → File → Share → Publish to web → CSV → paste the URL.

In [7]:
GOOGLE_SHEET_CSV_URL = None

staff_rows = [
    {"name": "Sara Nabil", "role": "Nurse", "ward": "ICU", "leave_balance": 8,
     "start_date": "2023-01-10", "on_shift_today": "yes", "isolation_clearance": "yes",
     "manager_email": "icu.lead@nilecare.example"},
    {"name": "Omar Tarek", "role": "Nurse", "ward": "ICU", "leave_balance": 1,
     "start_date": "2024-06-01", "on_shift_today": "yes", "isolation_clearance": "yes",
     "manager_email": "icu.lead@nilecare.example"},
    {"name": "Maya Fathy", "role": "Nurse", "ward": "ICU", "leave_balance": 12,
     "start_date": "2022-03-15", "on_shift_today": "yes", "isolation_clearance": "no",
     "manager_email": "icu.lead@nilecare.example"},
    {"name": "Karim Youssef", "role": "Nurse", "ward": "ER", "leave_balance": 5,
     "start_date": "2023-09-01", "on_shift_today": "yes", "isolation_clearance": "yes",
     "manager_email": "er.lead@nilecare.example"},
    {"name": "Hana Samir", "role": "Nurse", "ward": "ER", "leave_balance": 0,
     "start_date": "2026-08-15", "on_shift_today": "yes", "isolation_clearance": "no",
     "manager_email": "er.lead@nilecare.example"},
    {"name": "Yara Adel", "role": "Nurse", "ward": "Pediatrics", "leave_balance": 10,
     "start_date": "2021-11-20", "on_shift_today": "yes", "isolation_clearance": "no",
     "manager_email": "peds.lead@nilecare.example"},
    {"name": "Tamer Fouad", "role": "Doctor", "ward": "ER", "leave_balance": 15,
     "start_date": "2020-02-01", "on_shift_today": "no", "isolation_clearance": "yes",
     "manager_email": "er.lead@nilecare.example"},
    {"name": "Noha Essam", "role": "Nurse", "ward": "Pediatrics", "leave_balance": 3,
     "start_date": "2024-12-01", "on_shift_today": "yes", "isolation_clearance": "no",
     "manager_email": "peds.lead@nilecare.example"},
]

sheet_path = "nilecare_staff_roster.csv"
pd.DataFrame(staff_rows).to_csv(sheet_path, index=False)


def load_google_sheet(url=None, local_path=sheet_path):
    if url:
        print("Loading from published Google Sheet")
        return pd.read_csv(url)
    print("Loading local roster (same columns as a published Sheet)")
    return pd.read_csv(local_path)


staff_df = load_google_sheet(GOOGLE_SHEET_CSV_URL)
staff_df

Loading local roster (same columns as a published Sheet)


,name,role,ward,leave_balance,start_date,on_shift_today,isolation_clearance,manager_email
0,Sara Nabil,Nurse,ICU,8,2023-01-10,yes,yes,icu.lead@nilecare.example
1,Omar Tarek,Nurse,ICU,1,2024-06-01,yes,yes,icu.lead@nilecare.example
2,Maya Fathy,Nurse,ICU,12,2022-03-15,yes,no,icu.lead@nilecare.example
3,Karim Youssef,Nurse,ER,5,2023-09-01,yes,yes,er.lead@nilecare.example
4,Hana Samir,Nurse,ER,0,2026-08-15,yes,no,er.lead@nilecare.example
5,Yara Adel,Nurse,Pediatrics,10,2021-11-20,yes,no,peds.lead@nilecare.example
6,Tamer Fouad,Doctor,ER,15,2020-02-01,no,yes,er.lead@nilecare.example
7,Noha Essam,Nurse,Pediatrics,3,2024-12-01,yes,no,peds.lead@nilecare.example


## 7. Chunking, embeddings, FAISS

**This code does the following:** Splits the policy by section, turns each chunk into a vector with LangChain HuggingFaceEmbeddings, and stores the vectors in FAISS.

In [8]:
def sheet_to_text(df):
    lines = ["NileCare staff roster. Each line is one staff member."]
    for _, row in df.iterrows():
        lines.append(
            f"{row['name']} is a {row['role']} in {row['ward']}. "
            f"Leave balance {row['leave_balance']} days. "
            f"Start date {row['start_date']}. "
            f"On shift today: {row['on_shift_today']}. "
            f"Isolation clearance: {row['isolation_clearance']}. "
            f"Manager email: {row['manager_email']}."
        )
    return "\n".join(lines)


def chunk_policy(text):
    parts = re.split(r"(?=Section [A-D]\.)", text)
    return [p.strip() for p in parts if p.strip()]


policy_chunks = chunk_policy(pdf_text)
knowledge_chunks = policy_chunks + [sheet_to_text(staff_df)]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
matrix = np.array(embeddings.embed_documents(knowledge_chunks)).astype("float32")
index = faiss.IndexFlatL2(matrix.shape[1])
index.add(matrix)

print("Policy chunks:", len(policy_chunks))
print("FAISS vectors:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Policy chunks: 5
FAISS vectors: 6


## 8. Retriever chain (LangChain)

**This code does the following:** Wraps FAISS search in a LangChain `RunnableLambda`. The question becomes a vector. FAISS returns the closest chunks.

In [9]:
def retrieve_context(query, k=3):
    q = np.array([embeddings.embed_query(query)]).astype("float32")
    distances, ids = index.search(q, k)
    return [{"chunk_id": int(i), "distance": float(d), "text": knowledge_chunks[i]}
            for d, i in zip(distances[0], ids[0])]


retrieve_chain = RunnableLambda(lambda query: retrieve_context(query, k=3))

print(retrieve_chain.invoke("How many leave days can a probation staff member take?")[0]["text"][:220])

Section A. Annual Leave
Full-time staff receive 21 annual leave days per year.
A leave request is denied if the requested days are greater than the staff member remaining leave
balance.
Staff in the first 90 days of empl


## 9. Four LangChain tools

**This code does the following:** Wraps the four sheet functions as LangChain `StructuredTool` objects. The orchestrator calls `.invoke(...)`, not a raw Python guess.

| Tool | Job |
|------|-----|
| `get_staff` | One person from the roster |
| `check_leave_balance` | Remaining days vs requested days |
| `check_shift_coverage` | Would this leave break the ward minimum? |
| `get_department_contact` | Who to email |

In [10]:
TODAY = date(2026, 10, 11)
MIN_STAFF = {"ICU": 3, "ER": 2, "Pediatrics": 2}


def _find_staff(name):
    if not name:
        return None
    name = str(name).strip().lower()
    for _, row in staff_df.iterrows():
        if name in str(row["name"]).lower():
            return row.to_dict()
    return None


def get_staff(name: str) -> dict:
    """Look up one staff member in the NileCare roster."""
    row = _find_staff(name)
    if row is None:
        return {"found": False, "reason": f"No staff named '{name}' in the roster."}
    start = datetime.strptime(str(row["start_date"]), "%Y-%m-%d").date()
    days_employed = (TODAY - start).days
    return {
        "found": True,
        "name": row["name"],
        "role": row["role"],
        "ward": row["ward"],
        "leave_balance": int(row["leave_balance"]),
        "start_date": row["start_date"],
        "on_shift_today": row["on_shift_today"],
        "isolation_clearance": row["isolation_clearance"],
        "manager_email": row["manager_email"],
        "days_employed": days_employed,
        "on_probation": days_employed < 90
    }


def check_leave_balance(name: str, requested_days: int) -> dict:
    """Compare remaining leave days with the requested days."""
    staff = get_staff(name)
    if not staff.get("found"):
        return staff
    requested_days = int(requested_days)
    remaining = staff["leave_balance"]
    return {
        "name": staff["name"],
        "requested_days": requested_days,
        "leave_balance": remaining,
        "enough_balance": remaining >= requested_days,
        "on_probation": staff["on_probation"],
        "probation_limit_ok": (not staff["on_probation"]) or requested_days <= 2
    }


def check_shift_coverage(name: str) -> dict:
    """Check whether this leave would drop the ward below the minimum."""
    staff = get_staff(name)
    if not staff.get("found"):
        return staff
    ward = staff["ward"]
    on_shift = staff_df[(staff_df["ward"] == ward) & (staff_df["on_shift_today"].str.lower() == "yes")]
    current = int(len(on_shift))
    minimum = MIN_STAFF[ward]
    leaving = str(staff["on_shift_today"]).lower() == "yes"
    after = current - 1 if leaving else current
    return {
        "name": staff["name"],
        "ward": ward,
        "on_shift_today": staff["on_shift_today"],
        "current_on_shift": current,
        "minimum_required": minimum,
        "after_leave": after,
        "coverage_ok": after >= minimum
    }


def get_department_contact(name: str = "", ward: str = "") -> dict:
    """Return the department lead and email for a staff member or ward."""
    contacts = {
        "ICU": {"lead": "Head Nurse Mona Farid", "email": "icu.lead@nilecare.example"},
        "ER": {"lead": "Dr. Hany Aziz", "email": "er.lead@nilecare.example"},
        "Pediatrics": {"lead": "Head Nurse Nour Adel", "email": "peds.lead@nilecare.example"},
        "HR": {"lead": "Laila Hassan", "email": "hr@nilecare.example"},
    }
    if name:
        staff = get_staff(name)
        if staff.get("found"):
            info = contacts[staff["ward"]]
            return {"staff": staff["name"], "ward": staff["ward"], **info}
    if ward and ward in contacts:
        return {"ward": ward, **contacts[ward]}
    return {"ward": "HR", **contacts["HR"]}


get_staff_tool = StructuredTool.from_function(func=get_staff, name="get_staff")
leave_tool = StructuredTool.from_function(func=check_leave_balance, name="check_leave_balance")
coverage_tool = StructuredTool.from_function(func=check_shift_coverage, name="check_shift_coverage")
contact_tool = StructuredTool.from_function(func=get_department_contact, name="get_department_contact")

TOOL_CHAINS = {
    "get_staff": get_staff_tool,
    "check_leave_balance": leave_tool,
    "check_shift_coverage": coverage_tool,
    "get_department_contact": contact_tool,
}

print("LangChain tools:", list(TOOL_CHAINS))
print(get_staff_tool.invoke({"name": "Sara"}))

LangChain tools: ['get_staff', 'check_leave_balance', 'check_shift_coverage', 'get_department_contact']
{'found': True, 'name': 'Sara Nabil', 'role': 'Nurse', 'ward': 'ICU', 'leave_balance': 8, 'start_date': '2023-01-10', 'on_shift_today': 'yes', 'isolation_clearance': 'yes', 'manager_email': 'icu.lead@nilecare.example', 'days_employed': 1370, 'on_probation': False}


## 10. Memory

**This code does the following:** Stores the full chat (buffer memory) and the last staff member so a follow-up with no name still works.

In [11]:
class CareDeskMemory:
    def __init__(self):
        self.buffer = []
        self.summary = ""
        self.last_staff = None
        self.summary_every = 4

    def add(self, user_text, assistant_text, staff_name=None):
        self.buffer.append({"user": user_text, "assistant": assistant_text})
        if staff_name:
            self.last_staff = staff_name
        if len(self.buffer) % self.summary_every == 0:
            self.refresh_summary()

    def buffer_text(self):
        if not self.buffer:
            return "No previous conversation."
        lines = []
        for turn in self.buffer:
            lines.append("User: " + turn["user"])
            lines.append("Assistant: " + turn["assistant"])
        return "\n".join(lines)

    def refresh_summary(self):
        prompt = "Summarize this staff chat in 3 factual lines.\n" + self.buffer_text() + "\nSummary:"
        self.summary = generate(prompt, max_tokens=80)

    def reset(self):
        self.buffer = []
        self.summary = ""
        self.last_staff = None


memory = CareDeskMemory()
print("Memory ready.")

Memory ready.


## 11. Prompt chain + factual answer builder

**This code does the following:**  
- `quote_chain` is a LangChain PromptTemplate plus `generate`. It writes a short policy quote.  
- `build_answer` writes the user-facing answer from the decision and the tool facts. That stops the small model from saying Yes when the decision is ESCALATE, or from repeating the prompt.

In [12]:
rag_template = PromptTemplate.from_template(
    "Use only this context. Reply in one short sentence.\n"
    "Context: {context}\nQuestion: {question}\nShort answer:"
)

quote_chain = rag_template | RunnableLambda(lambda prompt: generate(str(prompt), max_tokens=60))


def build_answer(question, decision, reason, tool_results):
    q = question.lower()
    staff = tool_results.get("get_staff") or {}
    leave = tool_results.get("check_leave_balance") or {}
    coverage = tool_results.get("check_shift_coverage") or {}
    contact = tool_results.get("get_department_contact") or {}
    name = staff.get("name") or contact.get("staff") or "This staff member"

    if decision == "NOT FOUND":
        return (
            f"NOT FOUND. {reason} "
            "CareDesk answers only from the staff policy and the roster. "
            "It does not give medical treatment advice."
        )

    if "isolation" in q and staff.get("found"):
        flag = str(staff.get("isolation_clearance", "")).lower()
        if flag == "yes":
            return (
                f"ALLOW. {name} may enter isolation rooms. "
                "Isolation clearance on the roster is yes. "
                "Policy: only staff with isolation clearance may enter isolation rooms."
            )
        return (
            f"DENY. {name} may not enter isolation rooms. "
            f"Isolation clearance on the roster is {staff.get('isolation_clearance')}. "
            "Policy: only staff with isolation clearance may enter isolation rooms."
        )

    if decision == "DENY":
        extra = ""
        if leave:
            extra = f" {name} asked for {leave.get('requested_days')} day(s) and has {leave.get('leave_balance')} day(s) left."
        return f"DENY. {reason}.{extra}"

    if decision == "ESCALATE":
        extra = ""
        if coverage:
            extra = (
                f" {name} works in {coverage.get('ward')}. "
                f"On shift now: {coverage.get('current_on_shift')}. "
                f"Minimum: {coverage.get('minimum_required')}. "
                f"After this leave: {coverage.get('after_leave')}."
            )
        return f"ESCALATE. {reason}.{extra} Send the request to the department lead before approving."

    if decision == "ALLOW":
        extra = ""
        if leave:
            extra = f" {name} asked for {leave.get('requested_days')} day(s) and has {leave.get('leave_balance')} day(s) left."
        return f"ALLOW. {reason}.{extra}"

    if contact:
        return (
            f"INFO. Email {contact.get('lead')} at {contact.get('email')} "
            f"about {contact.get('staff', name)} in {contact.get('ward')}. "
            "This contact comes from the roster and the policy."
        )

    return f"{decision}. {reason}"

## 12. LangChain orchestrator

**This code does the following:** Builds one LangChain chain. Each step is a `RunnableLambda`. The pipe `|` is LangChain LCEL: the output of one step is the input of the next.

| Chain step | What it calls |
|------------|----------------|
| `parse` | extract name, days, tool list |
| `tools` | the 4 LangChain tools |
| `retrieve` | `retrieve_chain` + `quote_chain` |
| `decide` | the Python policy rules |
| `speak` | `build_answer` |
| `remember` | buffer memory + print the trace |

`desk.run(question)` starts this chain. Tests and chat use the same entry point.

In [13]:
def extract_days(text):
    match = re.search(r"(\d+)\s*day", text.lower())
    return int(match.group(1)) if match else None


def extract_staff_name(text):
    low = text.lower()
    for name in staff_df["name"].tolist():
        if name.lower() in low or name.split()[0].lower() in low.split():
            return name
    return None


def pick_tools(question):
    q = question.lower()
    names = []
    if any(w in q for w in ["leave", "day off", "days off", "vacation", "time off", "annual"]):
        names.extend(["get_staff", "check_leave_balance", "check_shift_coverage"])
    if any(w in q for w in ["contact", "email", "who should", "manager", "escalate", "lead"]):
        names.append("get_department_contact")
    if any(w in q for w in ["isolation", "clearance"]):
        names.append("get_staff")
    seen = []
    for n in names:
        if n not in seen:
            seen.append(n)
    return seen


def is_out_of_scope(question):
    q = question.lower()
    medical = ["diagnose", "treatment", "dose", "symptom", "heart attack",
               "prescribe", "surgery", "protocol for patient"]
    return any(w in q for w in medical)


def make_decision(question, tool_results):
    if is_out_of_scope(question):
        return "NOT FOUND", "This assistant does not answer medical treatment questions."

    leave = tool_results.get("check_leave_balance")
    coverage = tool_results.get("check_shift_coverage")
    staff = tool_results.get("get_staff")

    if staff and not staff.get("found"):
        return "NOT FOUND", "That name is not in the staff roster."

    if "isolation" in question.lower() and staff and staff.get("found"):
        if str(staff.get("isolation_clearance", "")).lower() == "yes":
            return "ALLOW", "This staff member has isolation clearance."
        return "DENY", "This staff member does not have isolation clearance."

    if leave:
        if not leave.get("enough_balance"):
            return "DENY", "Requested days are greater than the remaining leave balance."
        if leave.get("on_probation") and not leave.get("probation_limit_ok"):
            return "DENY", "Probation staff may take at most 2 leave days."
        if leave.get("on_probation"):
            return "ESCALATE", "Probation leave must be sent to the department lead."
        if coverage and not coverage.get("coverage_ok"):
            return "ESCALATE", "Approving this leave would drop the ward below the minimum."
        return "ALLOW", "Leave balance is enough and ward coverage stays safe."

    if tool_results.get("get_department_contact"):
        return "INFO", "Contact details come from the policy and the staff roster."

    return "INFO", "Answer from retrieved policy only."


def parse_step(state):
    question = state["question"]
    state["staff_name"] = extract_staff_name(question) or memory.last_staff
    state["requested_days"] = extract_days(question) or 1
    state["tool_names"] = pick_tools(question)
    return state


def tools_step(state):
    results = {}
    name = state.get("staff_name") or ""
    days = state.get("requested_days", 1)
    for tool_name in state.get("tool_names", []):
        tool = TOOL_CHAINS[tool_name]
        if tool_name == "get_staff":
            results[tool_name] = tool.invoke({"name": name})
        elif tool_name == "check_leave_balance":
            results[tool_name] = tool.invoke({"name": name, "requested_days": days})
        elif tool_name == "check_shift_coverage":
            results[tool_name] = tool.invoke({"name": name})
        elif tool_name == "get_department_contact":
            results[tool_name] = tool.invoke({"name": name})
    state["tool_results"] = results
    return state


def retrieve_step(state):
    question = state["question"]
    hits = retrieve_chain.invoke(question)
    context = "\n\n".join(h["text"] for h in hits)
    quote = quote_chain.invoke({"question": question, "context": context})
    if is_out_of_scope(question):
        quote = "Not found in document"
    elif llm_text_is_bad(quote):
        quote = hits[0]["text"][:280] if hits else "Not found in document"
    state["hits"] = hits
    state["policy_quote"] = quote
    return state


def decide_step(state):
    decision, reason = make_decision(state["question"], state.get("tool_results", {}))
    state["decision"] = decision
    state["reason"] = reason
    return state


def speak_step(state):
    state["answer"] = build_answer(
        state["question"],
        state["decision"],
        state["reason"],
        state.get("tool_results", {}),
    )
    return state


def remember_step(state):
    question = state["question"]
    answer = state["answer"]
    staff_name = state.get("staff_name")
    tool_results = state.get("tool_results", {})
    tool_names = state.get("tool_names", [])
    hits = state.get("hits", [])

    used_staff = None
    if staff_name and tool_results.get("get_staff", {}).get("found"):
        used_staff = tool_results["get_staff"]["name"]
    elif staff_name:
        used_staff = staff_name

    memory.add(question, answer, staff_name=used_staff)

    result = {
        "question": question,
        "decision": state["decision"],
        "reason": state["reason"],
        "tools_used": tool_names,
        "staff_in_memory": memory.last_staff,
        "retrieved_chunks": [h["chunk_id"] for h in hits],
        "policy_quote": state.get("policy_quote", ""),
        "tool_results": tool_results,
        "answer": answer,
    }

    if state.get("show_trace", True):
        print("=" * 60)
        print("QUESTION :", question)
        print("DECISION :", state["decision"], "|", state["reason"])
        print("TOOLS    :", tool_names or "none (RAG only)")
        print("CHUNKS   :", result["retrieved_chunks"])
        print("MEMORY   : last staff =", memory.last_staff, "| turns =", len(memory.buffer))
        print("-" * 60)
        print("ANSWER:\n" + answer)
        print("=" * 60)

    return result


caredesk_chain = (
    RunnableLambda(parse_step)
    | RunnableLambda(tools_step)
    | RunnableLambda(retrieve_step)
    | RunnableLambda(decide_step)
    | RunnableLambda(speak_step)
    | RunnableLambda(remember_step)
)


class CareDeskOrchestrator:
    def run(self, question, show_trace=True):
        return caredesk_chain.invoke({
            "question": question,
            "show_trace": show_trace,
        })


desk = CareDeskOrchestrator()
print("LangChain orchestrator ready.")
print("Chain: parse | tools | retrieve | decide | speak | remember")

LangChain orchestrator ready.
Chain: parse | tools | retrieve | decide | speak | remember


## 13. Live tests

**This code does the following:** Runs five questions in order. Test 4 has no name on purpose so memory must keep Sara.

| Test | Question | Expected |
|------|----------|----------|
| 1 | Tamer, 3 days | ALLOW |
| 2 | Omar, 3 days | DENY |
| 3 | Sara, 3 days | ESCALATE |
| 4 | Who should I email about her request? | INFO, memory = Sara |
| 5 | Heart-attack treatment | NOT FOUND |

In [14]:
memory.reset()

test_questions = [
    "Can Tamer Fouad take 3 days of leave?",
    "Can Omar Tarek take 3 days of leave?",
    "Can Sara Nabil take 3 days of leave?",
    "Who should I email about her request?",
    "What is the treatment protocol for a heart attack?"
]

results = []
for q in test_questions:
    results.append(desk.run(q))
    print()

QUESTION : Can Tamer Fouad take 3 days of leave?
DECISION : ALLOW | Leave balance is enough and ward coverage stays safe.
TOOLS    : ['get_staff', 'check_leave_balance', 'check_shift_coverage']
CHUNKS   : [1, 5, 2]
MEMORY   : last staff = Tamer Fouad | turns = 1
------------------------------------------------------------
ANSWER:
ALLOW. Leave balance is enough and ward coverage stays safe.. Tamer Fouad asked for 3 day(s) and has 15 day(s) left.

QUESTION : Can Omar Tarek take 3 days of leave?
DECISION : DENY | Requested days are greater than the remaining leave balance.
TOOLS    : ['get_staff', 'check_leave_balance', 'check_shift_coverage']
CHUNKS   : [1, 5, 2]
MEMORY   : last staff = Omar Tarek | turns = 2
------------------------------------------------------------
ANSWER:
DENY. Requested days are greater than the remaining leave balance.. Omar Tarek asked for 3 day(s) and has 1 day(s) left.

QUESTION : Can Sara Nabil take 3 days of leave?
DECISION : ESCALATE | Approving this leave w

## 14. Evaluation

**This code does the following:** Compares the five results to an answer key. Run this only after the test cell finishes.

In [15]:
expected = [
    {"question": "Can Tamer Fouad take 3 days of leave?", "decision": "ALLOW",
     "must_use_tools": ["get_staff", "check_leave_balance", "check_shift_coverage"], "memory_staff": "Tamer Fouad"},
    {"question": "Can Omar Tarek take 3 days of leave?", "decision": "DENY",
     "must_use_tools": ["check_leave_balance"], "memory_staff": "Omar Tarek"},
    {"question": "Can Sara Nabil take 3 days of leave?", "decision": "ESCALATE",
     "must_use_tools": ["check_shift_coverage"], "memory_staff": "Sara Nabil"},
    {"question": "Who should I email about her request?", "decision": "INFO",
     "must_use_tools": ["get_department_contact"], "memory_staff": "Sara Nabil"},
    {"question": "What is the treatment protocol for a heart attack?", "decision": "NOT FOUND",
     "must_use_tools": [], "memory_staff": "Sara Nabil"},
]

rows = []
for exp, got in zip(expected, results):
    rows.append({
        "question": exp["question"],
        "expected": exp["decision"],
        "got": got["decision"],
        "decision_ok": got["decision"] == exp["decision"],
        "tools_ok": all(t in got["tools_used"] for t in exp["must_use_tools"]),
        "memory_ok": got["staff_in_memory"] == exp["memory_staff"],
        "relevant": len(got["answer"]) > 0,
    })

eval_df = pd.DataFrame(rows)
if eval_df.empty:
    print("results is empty. Run the 5-question test cell first.")
else:
    score = eval_df[["decision_ok", "tools_ok", "memory_ok", "relevant"]].mean().mean()
    print("Evaluation table")
    display(eval_df)
    print(f"\nOverall score: {score:.0%}")

Evaluation table


,question,expected,got,decision_ok,tools_ok,memory_ok,relevant
0,Can Tamer Fouad take 3 days of leave?,ALLOW,ALLOW,True,True,True,True
1,Can Omar Tarek take 3 days of leave?,DENY,DENY,True,True,True,True
2,Can Sara Nabil take 3 days of leave?,ESCALATE,ESCALATE,True,True,True,True
3,Who should I email about her request?,INFO,INFO,True,True,True,True
4,What is the treatment protocol for a heart att...,NOT FOUND,NOT FOUND,True,True,True,True



Overall score: 100%


## 15. Interactive chat

**This code does the following:** Opens a live loop on the same LangChain chain. Type `quit` to stop, `reset` to clear memory.

Try:
- `Can Yara Adel take 2 days of leave?` → ESCALATE (Pediatrics would drop below 2)
- `Who should I email about that?` → Nour Adel, peds.lead@nilecare.example
- `Can Maya Fathy enter isolation rooms?` → DENY (clearance is no)

In [16]:
print("CareDesk chat")
print("Type quit to stop. Type reset to clear memory.\n")

while True:
    user = input("You: ").strip()
    if not user:
        continue
    if user.lower() in {"quit", "exit", "q"}:
        print("Chat closed.")
        break
    if user.lower() == "reset":
        memory.reset()
        print("Memory cleared.\n")
        continue
    desk.run(user)
    print()

CareDesk chat
Type quit to stop. Type reset to clear memory.

You: Can Yara Adel take 2 days of leave?
QUESTION : Can Yara Adel take 2 days of leave?
DECISION : ESCALATE | Approving this leave would drop the ward below the minimum.
TOOLS    : ['get_staff', 'check_leave_balance', 'check_shift_coverage']
CHUNKS   : [1, 5, 2]
MEMORY   : last staff = Yara Adel | turns = 6
------------------------------------------------------------
ANSWER:
ESCALATE. Approving this leave would drop the ward below the minimum.. Yara Adel works in Pediatrics. On shift now: 2. Minimum: 2. After this leave: 1. Send the request to the department lead before approving.

You: Who should I email about that?
QUESTION : Who should I email about that?
DECISION : INFO | Contact details come from the policy and the staff roster.
TOOLS    : ['get_department_contact']
CHUNKS   : [4, 5, 0]
MEMORY   : last staff = Yara Adel | turns = 7
------------------------------------------------------------
ANSWER:
INFO. Email Head Nur

## What this project is (30 seconds)

> CareDesk is a hospital **staff** knowledge assistant.  
> It reads a **policy PDF** and a **staff Google Sheet**.  
> It searches with **HuggingFace embeddings + FAISS**.  
> **LangChain** is the orchestrator: one chain of parse, tools, retrieve, decide, speak, remember.  
> The four tools are LangChain `StructuredTool` objects.  
> Python decides allow / deny / escalate. The spoken answer is built from those facts.  
> Buffer memory keeps follow-ups on the same staff member.

### CV line

> Built CareDesk, a LangChain-orchestrated RAG assistant over hospital policy PDFs and staff sheets, with FAISS retrieval, tool-checked leave decisions, and conversation memory.

### Rubric checklist

- [x] PDFs
- [x] Google Sheets (CSV publish format + live URL slot)
- [x] HuggingFace embeddings
- [x] FAISS
- [x] RAG question → retrieve → answer
- [x] Buffer memory + follow-ups
- [x] Testing and evaluation
- [x] LangChain orchestrator (LCEL chain)
- [x] LangChain tools + prompt chain